# Reproduce `openhands_star`

Generated from `<inference page: openhands_star x 23 instance(s)>`. The notebook replays inference + evaluation against the exact config snapshot that produced the original run, so a re-execution should produce a comparable `model_patch` (LLM determinism caveats notwithstanding).

- **Instances:** 23
- **Config id:** `openhands_star`


## 1. Setup

The notebook's `kernelspec.name = "evomas"` (see metadata at the bottom of the file) tells Jupyter / VSCode to auto-pick the interpreter `setup.ps1` / `setup.sh` registered for `~/.evomas-venv`. As a safety net the first cell also prepends the venv's site-packages to `sys.path` — so even if the kernel falls back to a generic Python 3 (different machine, no `setup.ps1` run), the evomas imports still resolve. Adjust `OLLAMA_BASE_URL` if your Ollama daemon isn't on the default host; `SWEBENCH_API_KEY` is only required by the remote-eval cell at the bottom.

### Picking the kernel in VSCode

If VSCode opens the notebook outside the EvoMas workspace (e.g. straight from `~/Downloads`), it won't auto-resolve the kernelspec and asks you to **Select Kernel**. Two-tier picker:

- **"Python Environments…"** lists raw Python interpreters discovered by the Python extension (system Python, conda envs, `.venv`/`venv` folders inside workspaces). `~/.evomas-venv` is outside the conventional discovery paths, so it does NOT show up here.
- **"Jupyter Kernel…"** lists registered Jupyter kernelspecs (`%APPDATA%\jupyter\kernels\*` on Windows, `~/.local/share/jupyter/kernels/*` on Linux/mac). This is where the EvoMas one lives — pick **"Python 3 (EvoMas)"** here. VSCode remembers the choice per-notebook so you only have to do it once.

If the entry doesn't appear there: `Ctrl+Shift+P` → **"Developer: Reload Window"** so the Jupyter extension re-scans kernelspecs, or run `jupyter kernelspec list` to confirm `evomas` is registered (if not, re-run `setup.ps1` / `setup.sh`).

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

# Defensive sys.path prepend: if the running kernel isn't the
# evomas-venv one (e.g. user opened the notebook on a fresh
# clone without running setup.ps1, or VSCode picked a generic
# Python 3), surface the venv's site-packages so `import
# evomas...` still resolves. Skipped when the active sys
# already points at the venv.
_venv = Path.home() / '.evomas-venv'
if _venv.is_dir() and str(_venv) not in sys.executable:
    for _sp in (_venv / 'Lib' / 'site-packages',
                _venv / 'lib' / 'site-packages'):
        if _sp.is_dir() and str(_sp) not in sys.path:
            sys.path.insert(0, str(_sp))

from evomas.core.workflow.runner import run as run_evomas
from evomas.utils.instances import fetch_swebench_instances

# Route Python `logging` records to BOTH the notebook output
# AND a per-run text log so `experiments/generate_report.py`
# can mine handoffs / tool calls / per-LLM-call tokens from
# the same lines the API matrix path writes. `force=True`
# overrides any prior basicConfig (e.g. from a stale kernel)
# so the format actually takes effect.
import logging
RUN_OUTPUT_DIR = Path('notebook-openhands_star').resolve()
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = RUN_OUTPUT_DIR / 'inference.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    force=True,
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
    ],
)
print(f'Mirroring inference logs to {LOG_FILE}')


Mirroring inference logs to C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-openhands_star\inference.log


### Environment variables

All run-time configuration the notebook needs lives in this cell — edit values here rather than chasing them through the code. Each assignment **overrides** whatever is in the environment / .env when the cell runs.

- **`OLLAMA_BASE_URL`** — where the Ollama daemon serves.
- **`SWEBENCH_API_KEY`** — required by the remote-eval cell (section 5) when running `--remote` against sb-cli. Local Docker harness runs don't need it.
- **`EVOMAS_INSTANCES`** — override the SWE-bench instance cache location. The next cell already searches sensible defaults; only set this if your cache is somewhere non-standard.
- **`GOOGLE_API_KEY` / `OPENAI_API_KEY`** — only needed if the inlined config picks a Gemini / OpenAI model instead of Ollama.


In [2]:
# Edit values here; each assignment overrides the inherited
# environment / .env. Uncomment the lines you need.
os.environ['OLLAMA_BASE_URL'] = 'http://192.168.1.50:11434'
# os.environ['SWEBENCH_API_KEY'] = 'swb_...'
# os.environ['EVOMAS_INSTANCES'] = '/path/to/swebench_instances.jsonl'
# os.environ['GOOGLE_API_KEY']   = '...'
# os.environ['OPENAI_API_KEY']   = '...'

# Echo the effective values (mask secrets) so you can verify the cell ran.
for _k in ('OLLAMA_BASE_URL', 'SWEBENCH_API_KEY', 'EVOMAS_INSTANCES',
           'GOOGLE_API_KEY', 'OPENAI_API_KEY'):
    _v = os.environ.get(_k, '')
    if not _v:
        print(f'  {_k:<18} <unset>')
    elif _k.endswith('_API_KEY'):
        print(f'  {_k:<18} {_v[:6]}***({len(_v)} chars)')
    else:
        print(f'  {_k:<18} {_v}')


  OLLAMA_BASE_URL    http://192.168.1.50:11434
  SWEBENCH_API_KEY   swb_QM***(56 chars)
  EVOMAS_INSTANCES   <unset>
  GOOGLE_API_KEY     AIzaSy***(39 chars)
  OPENAI_API_KEY     <unset>


## 2. Inlined config

Exact resolved config the original run used. Tweak hyperparameters here if you want to experiment with variations.

The cell below the config dict renders a mermaid diagram of the topology so you can see the agent-graph shape at a glance. The diagram is regenerated from `CONFIG['edges']` + `CONFIG['agents']` every time the cell runs, so edits to the dict above are reflected immediately.

In [3]:
CONFIG = {   'id': 'openhands_star',
    'description': 'OpenHands centralized star: AgentController hub dispatches to LocAgent, '
                   'CodeActAgent (main coder), ReadOnlyAgent, BrowsingAgent; DummyAgent acts as '
                   'the terminal sink. Every node is sourced from the OpenHands catalog.',
    'entry': 'controller',
    'end': 'finalizer',
    'edges': [   {'from': 'controller', 'to': 'locator'},
                 {'from': 'locator', 'to': 'controller'},
                 {'from': 'controller', 'to': 'coder'},
                 {'from': 'coder', 'to': 'controller'},
                 {'from': 'controller', 'to': 'reader'},
                 {'from': 'reader', 'to': 'controller'},
                 {'from': 'controller', 'to': 'browser'},
                 {'from': 'browser', 'to': 'controller'},
                 {'from': 'controller', 'to': 'finalizer'}],
    'agents': {   'controller': {   'class': 'Router',
                                    'variant': 'OpenHands:AgentController',
                                    'model': 'ollama/qwen3.5:9b',
                                    'think': True,
                                    'num_ctx': 2048,
                                    'stream': True,
                                    'temperature': 0,
                                    'top_k': 40,
                                    'top_p': 0.9,
                                    'min_p': 0,
                                    'repeat_penalty': 1.1,
                                    'repeat_last_n': 64,
                                    'seed': 0,
                                    'num_predict': 1024,
                                    'stop': [],
                                    'max_iters': 1,
                                    'prompts': {   'system': 'You dispatch the next worker in an '
                                                             'OpenHands-flavoured SWE-bench '
                                                             'pipeline. Critical path: locator '
                                                             '(LocAgent — finds source files) -> '
                                                             'coder (CodeActAgent — edits/patches '
                                                             'code) -> finalizer (emits the '
                                                             'workspace diff). `reader` and '
                                                             '`browser` are optional helpers and '
                                                             'are never selected here. Answer with '
                                                             'EXACTLY ONE word.',
                                                   'user': 'Apply these rules IN ORDER. Stop at '
                                                           'the first rule that matches.\n'
                                                           '\n'
                                                           '1. If `locator` output is EMPTY -> '
                                                           'answer: locator\n'
                                                           '2. If `coder` output is EMPTY -> '
                                                           'answer: coder\n'
                                                           '3. Otherwise -> answer: finalizer\n'
                                                           '\n'
                                                           '--- pipeline state (empty between '
                                                           '`<<<` and `>>>` means that worker '
                                                           "hasn't run yet) ---\n"
                                                           'locator output:  <<<{locator}>>>\n'
                                                           'coder output:    <<<{coder}>>>\n'
                                                           'reader output:   <<<{reader}>>>\n'
                                                           'browser output:  <<<{browser}>>>\n'
                                                           '--- end state ---\n'
                                                           '\n'
                                                           'Answer with one word from: locator, '
                                                           'coder, finalizer.'}},
                  'locator': {   'class': 'Locator',
                                 'variant': 'OpenHands:LocAgent',
                                 'model': 'ollama/qwen3.5:9b',
                                 'think': True,
                                 'num_ctx': 8192,
                                 'stream': True,
                                 'temperature': 0.2,
                                 'top_k': 40,
                                 'top_p': 0.9,
                                 'min_p': 0,
                                 'repeat_penalty': 1.1,
                                 'repeat_last_n': 64,
                                 'seed': 0,
                                 'num_predict': 512,
                                 'stop': ['</files>'],
                                 'max_iters': 6},
                  'coder': {   'class': 'Base agent',
                               'variant': 'OpenHands:CodeActAgent',
                               'model': 'ollama/qwen3.5:9b',
                               'think': True,
                               'num_ctx': 8192,
                               'stream': True,
                               'temperature': 0.3,
                               'top_k': 40,
                               'top_p': 0.9,
                               'min_p': 0,
                               'repeat_penalty': 1.1,
                               'repeat_last_n': 64,
                               'seed': 0,
                               'num_predict': 1024,
                               'stop': [],
                               'max_iters': 6,
                               'prompts': {   'user': '## Task\n'
                                                      '{issue}\n'
                                                      '\n'
                                                      '## Workspace\n'
                                                      '{workspace}\n'
                                                      '\n'
                                                      'Locate the buggy code and FIX IT BY CALLING '
                                                      '`StrReplaceEditorTool` with '
                                                      "command='str_replace'. Do NOT just inspect "
                                                      'the file with `ViewTool` — you MUST '
                                                      'actually edit it. The pipeline reads `git '
                                                      'diff` against the workspace as the final '
                                                      'patch, so an unedited workspace produces NO '
                                                      'PATCH and the run fails.\n'
                                                      '\n'
                                                      'Workflow: (1) `ViewTool` once to see the '
                                                      'buggy line(s), (2) `StrReplaceEditorTool` '
                                                      "with command='str_replace' to apply the "
                                                      'fix, (3) stop — the finalizer takes it from '
                                                      'here.'},
                               'tools': [{'name': 'ViewTool'}, {'name': 'StrReplaceEditorTool'}]},
                  'reader': {   'class': 'Helper/Proxy',
                                'variant': 'OpenHands:ReadOnlyAgent',
                                'model': 'ollama/qwen3.5:9b',
                                'think': True,
                                'num_ctx': 4096,
                                'stream': True,
                                'temperature': 0,
                                'top_k': 40,
                                'top_p': 0.9,
                                'min_p': 0,
                                'repeat_penalty': 1.1,
                                'repeat_last_n': 64,
                                'seed': 0,
                                'num_predict': 512,
                                'stop': [],
                                'max_iters': 4},
                  'browser': {   'class': 'Helper/Proxy',
                                 'variant': 'OpenHands:BrowsingAgent',
                                 'model': 'ollama/qwen3.5:9b',
                                 'think': True,
                                 'num_ctx': 4096,
                                 'stream': True,
                                 'temperature': 0,
                                 'top_k': 40,
                                 'top_p': 0.9,
                                 'min_p': 0,
                                 'repeat_penalty': 1.1,
                                 'repeat_last_n': 64,
                                 'seed': 0,
                                 'num_predict': 512,
                                 'stop': [],
                                 'max_iters': 4},
                  'finalizer': {   'class': 'Helper/Proxy',
                                   'variant': 'OpenHands:DummyAgent',
                                   'model': 'ollama/qwen3.5:9b',
                                   'think': True,
                                   'num_ctx': 4096,
                                   'stream': True,
                                   'temperature': 0,
                                   'top_k': 40,
                                   'top_p': 0.9,
                                   'min_p': 0,
                                   'repeat_penalty': 1.1,
                                   'repeat_last_n': 64,
                                   'seed': 0,
                                   'num_predict': 512,
                                   'stop': [],
                                   'max_iters': 4}}}

In [4]:
from IPython.display import Markdown, display

def _topology_mermaid(cfg):
    """Render the topology as a Mermaid flowchart.

    Mirrors what the topology page's cytoscape canvas shows:
    virtual START/END boundary nodes, one node per agent with
    its class as a second-line label, edges directed left-to-
    right. Renders inline in Jupyter Lab + VSCode Jupyter; if
    the cell falls back to plain text the source stays readable.
    """
    lines = ['graph LR']
    lines.append('    START((START))')
    lines.append('    END((END))')
    for name, block in (cfg.get('agents') or {}).items():
        cls = (block or {}).get('class', '') or ''
        label = f'{name}<br/><i>{cls}</i>' if cls else name
        # Backticks would break the mermaid parser; strip them
        # defensively. Class names never contain them today,
        # this is just future-proofing.
        label = label.replace('`', '')
        lines.append(f'    {name}["{label}"]')
    entry = cfg.get('entry') or ''
    if entry:
        lines.append(f'    START --> {entry}')
    for e in (cfg.get('edges') or []):
        if isinstance(e, dict) and e.get('from') and e.get('to'):
            lines.append(f'    {e["from"]} --> {e["to"]}')
    end_field = cfg.get('end')
    ends = (
        [end_field] if isinstance(end_field, str) and end_field
        else list(end_field or [])
    )
    # Only emit `→ END` for nodes with no outgoing edges (the
    # same wiring rule `graph_builder.py` uses). Hub-in-end
    # nodes with outgoing edges don't get the static edge.
    out_sources = {e.get('from') for e in (cfg.get('edges') or [])
                   if isinstance(e, dict)}
    for n in ends:
        if n and n not in out_sources:
            lines.append(f'    {n} --> END')
    return '\n'.join(lines)

display(Markdown('```mermaid\n' + _topology_mermaid(CONFIG) + '\n```'))


```mermaid
graph LR
    START((START))
    END((END))
    controller["controller<br/><i>Router</i>"]
    locator["locator<br/><i>Locator</i>"]
    coder["coder<br/><i>Base agent</i>"]
    reader["reader<br/><i>Helper/Proxy</i>"]
    browser["browser<br/><i>Helper/Proxy</i>"]
    finalizer["finalizer<br/><i>Helper/Proxy</i>"]
    START --> controller
    controller --> locator
    locator --> controller
    controller --> coder
    coder --> controller
    controller --> reader
    reader --> controller
    controller --> browser
    browser --> controller
    controller --> finalizer
    finalizer --> END
```

## 3. Instances

Self-contained: the notebook regenerates its own instances from zero each run. SWE-bench rows get pulled fresh from HuggingFace (cached under `~/.cache/huggingface`); custom rows are reconstructed from the minimal inputs the user added via the Inference page's `+ Custom` modal.

In [5]:
INSTANCE_IDS = [   'sqlfluff__sqlfluff-1625',
    'sqlfluff__sqlfluff-2419',
    'sqlfluff__sqlfluff-1733',
    'sqlfluff__sqlfluff-1517',
    'sqlfluff__sqlfluff-1763',
    'marshmallow-code__marshmallow-1359',
    'marshmallow-code__marshmallow-1343',
    'pvlib__pvlib-python-1707',
    'pvlib__pvlib-python-1072',
    'pvlib__pvlib-python-1606',
    'pvlib__pvlib-python-1854',
    'pvlib__pvlib-python-1154',
    'pylint-dev__astroid-1978',
    'pylint-dev__astroid-1333',
    'pylint-dev__astroid-1196',
    'pylint-dev__astroid-1866',
    'pylint-dev__astroid-1268',
    'pyvista__pyvista-4315',
    'pydicom__pydicom-1694',
    'pydicom__pydicom-1413',
    'pydicom__pydicom-901',
    'pydicom__pydicom-1139',
    'pydicom__pydicom-1256']


In [6]:
# Pull plan for SWE-bench rows: `{(subset, split): [ids]}`.
# At runtime the cell below calls `fetch_swebench_instances`
# per group and filters down to just these IDs.
SWEBENCH_GROUPS = {   ('lite', 'dev'): [   'sqlfluff__sqlfluff-1625',
                         'sqlfluff__sqlfluff-2419',
                         'sqlfluff__sqlfluff-1733',
                         'sqlfluff__sqlfluff-1517',
                         'sqlfluff__sqlfluff-1763',
                         'marshmallow-code__marshmallow-1359',
                         'marshmallow-code__marshmallow-1343',
                         'pvlib__pvlib-python-1707',
                         'pvlib__pvlib-python-1072',
                         'pvlib__pvlib-python-1606',
                         'pvlib__pvlib-python-1854',
                         'pvlib__pvlib-python-1154',
                         'pylint-dev__astroid-1978',
                         'pylint-dev__astroid-1333',
                         'pylint-dev__astroid-1196',
                         'pylint-dev__astroid-1866',
                         'pylint-dev__astroid-1268',
                         'pyvista__pyvista-4315',
                         'pydicom__pydicom-1694',
                         'pydicom__pydicom-1413',
                         'pydicom__pydicom-901',
                         'pydicom__pydicom-1139',
                         'pydicom__pydicom-1256']}


In [7]:
# Custom-instance inputs (no upstream — added locally via the
# Inference page's `+ Custom` modal). Notebook reconstructs the
# row dict from these fields; nothing else is needed.
CUSTOM_ROWS = []


In [8]:
# Materialise SWE-bench + custom rows into one JSONL the
# runner consumes. Re-uses `RUN_OUTPUT_DIR` from the setup
# cell so inference.log + prediction JSONL share one folder.
output_dir = RUN_OUTPUT_DIR
output_path = output_dir / 'prediction-openhands_star.jsonl'
INSTANCES_PATH = output_dir / 'instances.jsonl'

selected = []
for (subset, split), ids in SWEBENCH_GROUPS.items():
    print(f'Fetching {len(ids)} {subset}/{split} row(s) from HuggingFace…')
    selected.extend(fetch_swebench_instances(subset, split, instance_ids=ids))
selected.extend(CUSTOM_ROWS)

with INSTANCES_PATH.open('w', encoding='utf-8') as _fh:
    for _row in selected:
        _fh.write(json.dumps(_row, ensure_ascii=False) + '\n')
print(f'Wrote {len(selected)} instance row(s) -> {INSTANCES_PATH}')

_have = {i['instance_id'] for i in selected}
missing = [iid for iid in INSTANCE_IDS if iid not in _have]
if missing:
    print('Missing rows (id not found in HF or in CUSTOM_ROWS):', missing)
print(f'Ready to run {len(selected)} instance(s).')


Fetching 23 lite/dev row(s) from HuggingFace…


C:\Users\XF\.evomas-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-04 04:46:41,777 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"


2026-06-04 04:46:41,779 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-06-04 04:46:41,796 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SWE-bench/SWE-bench_Lite/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/README.md "HTTP/1.1 200 OK"


...

2026-06-04 04:46:42,773 [INFO] httpx: HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=SWE-bench/SWE-bench_Lite "HTTP/1.1 200 OK"


2026-06-04 04:46:42,909 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/datasets/SWE-bench/SWE-bench_Lite/tree/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/data?recursive=true&expand=false "HTTP/1.1 200 OK"


2026-06-04 04:46:43,042 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/datasets/SWE-bench/SWE-bench_Lite/tree/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e?recursive=false&expand=false "HTTP/1.1 200 OK"


2026-06-04 04:46:43,183 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/dataset_infos.json "HTTP/1.1 404 Not Found"


Wrote 23 instance row(s) -> C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-openhands_star\instances.jsonl
Ready to run 23 instance(s).


## 4. Inference

Re-runs the EvoMas workflow for each instance with the inlined config. All notebook-produced artefacts (prediction JSONL, evaluation reports, custom-instance sidecar) land under one per-run folder at `notebook-openhands_star/` so they stay grouped together and don't mix with UI/CLI runs in the repo's `results/` tree.

In [9]:
from evomas.exceptions.errors import OllamaMemoryError

# `output_dir` + `output_path` were created in the instances cell above.
predictions = []
with open(output_path, 'w', encoding='utf-8') as out:
    for inst in selected:
        iid = inst['instance_id']
        print(f'--- {iid} ---')
        try:
            patch = run_evomas(inst, config=CONFIG)
        except OllamaMemoryError as exc:
            print(f'Ollama OOM; aborting: {exc}')
            break
        except Exception as exc:
            print(f'run failed on {iid}: {exc}')
            patch = ''
        rec = {
            'instance_id': iid,
            'model_patch': patch,
            'model_name_or_path': 'evomas-notebook',
        }
        predictions.append(rec)
        out.write(json.dumps(rec) + '\n')
print(f'Wrote {len(predictions)} prediction(s) to {output_path}.')


2026-06-04 04:46:43,274 [WARNING] weave.trace.op: Warning: Traces will not be logged. Call weave.init to log your traces to a project.
 (subsequent messages of this type will be suppressed)


2026-06-04 04:46:43,275 [INFO] evomas.core.workflow.runner: === running sqlfluff__sqlfluff-1625 with inline config (id=openhands_star) ===


--- sqlfluff__sqlfluff-1625 ---


2026-06-04 04:46:43,635 [INFO] evomas.utils.workspace: reusing workspace at C:\Users\XF\AppData\Local\Temp\claude\evomas_workspace\sqlfluff__sqlfluff-1625 (HEAD=14e1a23a3166b9a645a16de96f694c77a5d4abb7)


2026-06-04 04:46:43,953 [INFO] evomas.core.workflow.runner: graph runtime: 6 agents x 2 max revisits => recursion_limit=12


...

2026-06-04 08:02:10,257 [INFO] evomas.models.langchain_ollama_model: [finalizer] tokens in=1170 out=103 total=1273


2026-06-04 08:02:10,258 [INFO] evomas.agents.finalizer: [finalizer] no tool calls — stopping loop


2026-06-04 08:02:10,316 [WARNING] evomas.core.workflow.runner: pydicom__pydicom-1256 produced empty patch (workspace clean AND state['finalizer'] not a diff)


2026-06-04 08:02:10,317 [INFO] evomas.core.workflow.runner: === pydicom__pydicom-1256 done: 0-char patch | tokens in=87092 out=7372 total=94464 ===


Wrote 23 prediction(s) to C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-openhands_star\prediction-openhands_star.jsonl.


## 5. Evaluation

Runs `scripts/evaluation/run_swebench_evaluation.py` — the evaluator chosen at notebook-generation time (the `--evaluator` flag on `evomas notebook`). Forwards through the unified evaluator CLI contract: `--predictions / --instances / --report-dir / --run-id / --model`. Output lands at `<report-dir>/{model}.{run-id}.json` plus per-instance folders under `<report-dir>/logs/run_evaluation/<run-id>/<model>/<instance>/`.

This evaluator is marked `needs_wsl=True`, so on Windows the script is wrapped through WSL (the harness is POSIX-only).

In [10]:
# Evaluator baked at notebook-gen time (--evaluator on `evomas notebook`).
EVALUATOR_STEM = 'run_swebench_evaluation'
EVALUATOR_NEEDS_WSL = True

first = selected[0] if selected else None
SUBSET = (first or {}).get('subset', 'lite')
SPLIT  = (first or {}).get('split',  'dev')

# All eval artifacts land under `output_dir` (alongside
# instances.jsonl + prediction-*.jsonl).
eval_report_dir = output_dir

import platform
from evomas.paths import BASE_DIR as _BASE_DIR
_script = _BASE_DIR / 'scripts' / 'evaluation' / f'{EVALUATOR_STEM}.py'
if EVALUATOR_NEEDS_WSL and platform.system() == 'Windows':
    from evomas.utils.paths import to_wsl
    cmd = [
        'wsl', '--', 'python3', to_wsl(str(_script)),
        '--predictions', to_wsl(str(output_path)),
        '--instances',   to_wsl(str(INSTANCES_PATH)),
        '--report-dir',  to_wsl(str(eval_report_dir)),
        '--run-id',      f'notebook-{SUBSET}-{SPLIT}',
        '--model',       'evomas-notebook',
    ]
else:
    cmd = [
        sys.executable, str(_script),
        '--predictions', str(output_path),
        '--instances',   str(INSTANCES_PATH),
        '--report-dir',  str(eval_report_dir),
        '--run-id',      f'notebook-{SUBSET}-{SPLIT}',
        '--model',       'evomas-notebook',
    ]
print(f'Evaluating via {EVALUATOR_STEM}.py')
print('+ ' + ' '.join(cmd))

eval_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, encoding='utf-8', errors='replace',
)
assert eval_proc.stdout is not None
for line in eval_proc.stdout:
    print(line, end='')
eval_proc.wait()
print(f'\n[evaluation finished with exit code {eval_proc.returncode}]')

# Surface the per-instance artifacts the evaluator wrote.
logs_root = eval_report_dir / 'logs' / 'run_evaluation'
if logs_root.is_dir():
    print('\nPer-instance artifacts:')
    for inst_dir in sorted(logs_root.rglob('*/')):
        if (inst_dir / 'report.json').is_file():
            print(f'  {inst_dir}')
for summary in sorted(eval_report_dir.glob('*.json')):
    print(f'Summary: {summary}')


Evaluating via run_swebench_evaluation.py
+ wsl -- python3 /mnt/c/Users/XF/Desktop/TFG/EvoMas/scripts/evaluation/run_swebench_evaluation.py --predictions /mnt/c/Users/XF/Desktop/TFG/EvoMas/notebooks/notebook-openhands_star/prediction-openhands_star.jsonl --instances /mnt/c/Users/XF/Desktop/TFG/EvoMas/notebooks/notebook-openhands_star/instances.jsonl --report-dir /mnt/c/Users/XF/Desktop/TFG/EvoMas/notebooks/notebook-openhands_star --run-id notebook-lite-dev --model evomas-notebook
2026-06-04 08:02:10,473 - INFO - Bucket lite/dev -> 23 row(s), run_id=notebook-lite-dev


2026-06-04 08:02:10,622 - INFO - Running evaluation with 8 workers on /mnt/c/Users/XF/Desktop/TFG/EvoMas/notebooks/notebook-openhands_star/_tmp_predictions/_bucket_lite_dev.jsonl (run_id=notebook-lite-dev, dataset=SWE-bench/SWE-bench_Lite, split=dev, force=True, cache_level=env, force_rebuild=False, python=/mnt/c/Users/XF/Desktop/TFG/EvoMas/SWE-bench/venv/bin/python, cwd=/mnt/c/Users/XF/Desktop/TFG/EvoMas/notebooks/notebook-openhands_star)


<frozen runpy>:128: RuntimeWarning: 'swebench.harness.run_evaluation' found in sys.modules after import of package 'swebench.harness', but prior to execution of 'swebench.harness.run_evaluation'; this may result in unpredictable behaviour


2026-06-04 08:02:29,435 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-04 08:02:29,436 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-04 08:02:29,452 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SWE-bench/SWE-bench_Lite/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/README.md "HTTP/1.1 200 OK"
2026-06-04 08:02:29,580 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite/resolve/69611d31007e1c6731db8bd5b5c3f2d33f5bab6e/SWE-bench_Lite.py "HTTP/1.1 404 Not Found"


2026-06-04 08:02:29,888 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/SWE-bench/SWE-bench_Lite/SWE-bench/SWE-bench_Lite.py "HTTP/1.1 404 Not Found"


...

Evaluation:   0%|          | 0/3 [00:27<?, ?it/s, ✓=0, ✖=1, error=0]


Evaluation:  33%|███▎      | 1/3 [00:36<00:55, 27.77s/it, ✓=0, ✖=2, error=0]


Evaluation: 100%|██████████| 3/3 [00:46<00:00, 15.46s/it, ✓=0, ✖=3, error=0]


All instances run.
Cleaning cached images...
Removed 0 images.
Total instances: 23
Instances submitted: 23
Instances completed: 3
Instances incomplete: 0
Instances resolved: 0
Instances unresolved: 3
Instances with empty patches: 20
Instances with errors: 0
Unstopped containers: 0
Unremoved images: 6
Report written to evomas-notebook.notebook-lite-dev.json



[evaluation finished with exit code 0]

Per-instance artifacts:
  C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-openhands_star\logs\run_evaluation\notebook-lite-dev\evomas-notebook\pydicom__pydicom-901
  C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-openhands_star\logs\run_evaluation\notebook-lite-dev\evomas-notebook\sqlfluff__sqlfluff-1517
  C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-openhands_star\logs\run_evaluation\notebook-lite-dev\evomas-notebook\sqlfluff__sqlfluff-1625
Summary: C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-openhands_star\evomas-notebook.notebook-lite-dev.json
